In [ ]:
import numpy as np
import pandas as pd
from scipy.ndimage import label
import os
import re
from google.colab import drive

In [ ]:
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
basepath = "/content/drive/MyDrive/DATASIGNALPROCESSING"  # change this to your own data folder
input_path = f"{basepath}/Data/04_CHB_MIT_WPE_Features_final_std"
summaries_path = f"{basepath}/Data/summaries"

In [ ]:
CONFIG = {
    'STEP_DUR': 2.5,
    'EPOCH_DUR': 5.0,
    'PRE_ICTAL_WIN': 30 * 60,
    'SPH_WIN': 5 * 60,
    'POST_ICTAL_GAP': 4 * 3600
}

WARMUP_CUTOFF = 480   

In [ ]:
def generate_exact_report():
    try:
        y_all = np.load(f"{input_path}/y_baseline_std.npy")
        g_all = np.load(f"{input_path}/groups_baseline_std.npy")
    except FileNotFoundError:
        print("Error, files not found")
        return

    patients = np.unique(g_all)
    report_rows = []
    for p_id in range(1, 25): 
        p_name = f"chb{p_id:02d}"

        if p_id in patients:
            mask = (g_all == p_id)
            y_p = y_all[mask]
            labeled_array, num_seizures = label(y_p)
            pre_count = np.sum(y_p == 1)
            inter_count = np.sum(y_p == 0)
            if pre_count > 0 and num_seizures == 0:
                num_seizures = 1 
            report_rows.append({
                "Patient": p_name,
                "Status": "Included",
                "Seizures_Counted": int(num_seizures),
                "Pre_ictal_Epochs": int(pre_count),
                "Inter_ictal_Epochs": int(inter_count)
            })
        else:
            report_rows.append({
                "Patient": p_name,
                "Status": "EXCLUDED",
                "Seizures_Counted": 0,
                "Pre_ictal_Epochs": 0,
                "Inter_ictal_Epochs": 0
            })

    df = pd.DataFrame(report_rows)
    mask_included = (df["Status"] == "Included") & (df["Pre_ictal_Epochs"] >= 480)
    print(df.to_string(index=False))
    print("\n" + "="*50)
    df.loc[(df["Pre_ictal_Epochs"] < 480) & (df["Pre_ictal_Epochs"] > 0), "Status"] = "Excluded (Too short)"
    df.loc[(df["Pre_ictal_Epochs"] == 0), "Status"] = "Excluded (No Pre-ictal)"
    total_seizures = df[df["Pre_ictal_Epochs"] >= 480]["Seizures_Counted"].sum()
    print(f"Total crisis (per AUC): {total_seizures}")
    print("="*50)

    return df

df_exact = generate_exact_report()

In [ ]:
y_all = np.load(f"{input_path}/y_baseline_std.npy").astype(np.int32)
g_all = np.load(f"{input_path}/groups_baseline_std.npy").astype(np.int32)

rows = []
for p_id in range(1, 25):
    p_name = f"chb{p_id:02d}"
    mask = (g_all == p_id)

    if not np.any(mask):
        rows.append({
            "Patient": p_name,
            "Total_Epochs": 0,
            "Preictal_Epochs_Total": 0,
            "Interictal_Epochs_Total": 0,
            "Seizures_After_Labeling": 0,
            "Preictal_Epochs_Eval": np.nan,
            "Interictal_Epochs_Eval": np.nan,
            "Seizures_In_Eval": np.nan,
            "Classes_In_Eval": 0,
            "Included_in_AUC": False,
            "Exclusion_Reason": "Not present in final dataset (no valid data at feature-extraction stage)"
        })
        continue

    y_p = y_all[mask]
    total_epochs = len(y_p)
    pre_total = int(np.sum(y_p == 1))
    inter_total = int(np.sum(y_p == 0))
    _, n_seizures_total = label(y_p == 1)
    y_eval = y_p[WARMUP_CUTOFF:] 

    if len(y_eval) == 0:
        classes_in_eval = 0
        pre_eval = inter_eval = 0
        n_seizures_eval = 0
    else:
        classes_in_eval = len(np.unique(y_eval))
        pre_eval = int(np.sum(y_eval == 1))
        inter_eval = int(np.sum(y_eval == 0))
        _, n_seizures_eval = label(y_eval == 1)

    included = classes_in_eval >= 2

    if included:
        reason = "-"
    elif pre_total == 0:
        reason = "No preictal epochs annotated (0 valid seizures after SPH/postictal exclusion)"
    else:
        reason = f"Preictal block(s) entirely within first {WARMUP_CUTOFF} epochs (~20 min warm-up) -> single-class eval segment"

    rows.append({
        "Patient": p_name,
        "Total_Epochs": total_epochs,
        "Preictal_Epochs_Total": pre_total,
        "Interictal_Epochs_Total": inter_total,
        "Seizures_After_Labeling": int(n_seizures_total),
        "Preictal_Epochs_Eval": pre_eval,
        "Interictal_Epochs_Eval": inter_eval,
        "Seizures_In_Eval": int(n_seizures_eval),
        "Classes_In_Eval": classes_in_eval,
        "Included_in_AUC": included,
        "Exclusion_Reason": reason
    })

df_final = pd.DataFrame(rows)
pd.set_option("display.width", 200)
print(df_final.to_string(index=False))

print("\n" + "="*70)
included_df = df_final[df_final["Included_in_AUC"] == True]
excluded_df = df_final[df_final["Included_in_AUC"] == False]

print(f"Pazienti totali nella coorte: {len(df_final)}")
print(f"Pazienti valutabili (inclusi in AUC): {len(included_df)}")
print(f"Pazienti esclusi: {len(excluded_df)} -> {excluded_df['Patient'].tolist()}")
print(f"Totale crisi valide (dopo warm-up, usate per l'AUC aggregata): {included_df['Seizures_In_Eval'].sum()}")
print(f"Totale epoche pre-ictali usate in valutazione: {included_df['Preictal_Epochs_Eval'].sum()}")
print(f"Totale epoche inter-ictali usate in valutazione: {included_df['Interictal_Epochs_Eval'].sum()}")
print("="*70)

df_final.to_csv(f"{basepath}/Output/dataset_exact_report.csv", index=False)

In [ ]:
def count_original_seizures(directory):
    print(f"Analisi dei file summary in: {directory}\n")

    patient_counts = []
    seizure_pattern = re.compile(r"Number of Seizures in File:\s*(\d+)", re.IGNORECASE)
    for i in range(1, 25):
        p_id = f"chb{i:02d}"
        filename = f"{p_id}-summary.txt"
        file_path = os.path.join(directory, filename)

        total_seizures = 0

        if os.path.exists(file_path):
            try:
                with open(file_path, 'r', encoding='latin-1') as f:
                    content = f.read()
                    matches = seizure_pattern.findall(content)
                    total_seizures = sum(int(m) for m in matches)

                patient_counts.append({
                    "Patient": p_id,
                    "Original_Seizures": total_seizures,
                    "Status": "File Found"
                })
            except Exception as e:
                patient_counts.append({
                    "Patient": p_id,
                    "Original_Seizures": 0,
                    "Status": f"Error: {str(e)}"
                })
        else:
            patient_counts.append({
                "Patient": p_id,
                "Original_Seizures": 0,
                "Status": "Summary Not Found"
            })

    # creating DataFrame
    df = pd.DataFrame(patient_counts)
    
    print("CONTEGGIO CRISI ORIGINALI")
    print(df.to_string(index=False))
    print("\n" + "="*40)
    print(f"TOTALE CRISI NEL DATABASE: {df['Original_Seizures'].sum()}")
    print("="*40)

    return df

In [ ]:
# exec
df_original = count_original_seizures(summaries_path)